In [1]:
# Parameters
base_path = "C:\\Users\\ander\\OneDrive\\TCC_USP"
run_id = "20251117_170659"


In [2]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "01_preprocessing"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


BASE_PATH: C:\Users\ander\OneDrive\TCC_USP
PROC_PATH: C:\Users\ander\OneDrive\TCC_USP\data_processed


In [3]:
# Legacy Colab mount removido; paths definidos no setup.


In [4]:
# 3. Carregar IBOV
import pandas as pd, os
ibov_file = os.path.join(RAW_PATH, "ibovespa.csv")
assert os.path.exists(ibov_file), f"Arquivo nÃ£o encontrado: {ibov_file}"
df_ibov = pd.read_csv(ibov_file)
df_ibov["data"] = pd.to_datetime(df_ibov["data"])
print("IBOV:", df_ibov.shape)
display(df_ibov.head())

IBOV: (20, 2)


,data,close
0,2024-01-02,130749.73
1,2024-01-03,130673.55
2,2024-01-04,131624.44
3,2024-01-05,133734.42
4,2024-01-08,133528.27


In [5]:
# 4. Carregar NotÃ­cias
news_file = os.path.join(RAW_PATH, "noticias_exemplo.csv")
assert os.path.exists(news_file), f"Arquivo nÃ£o encontrado: {news_file}"
df_news = pd.read_csv(news_file)
df_news["data"] = pd.to_datetime(df_news["data"])
print("NEWS:", df_news.shape)
display(df_news.head())

NEWS: (10, 5)


,data,fonte,titulo,texto,sentimento
0,2024-01-02,exemplo,Titulo 1,Ibovespa sobe com otimismo no mercado internac...,1
1,2024-01-03,exemplo,Titulo 2,Bolsa cai após dados fracos da economia chinesa.,0
2,2024-01-04,exemplo,Titulo 3,Petróleo em alta puxa ações da Petrobras para ...,1
3,2024-01-05,exemplo,Titulo 4,Queda do dólar beneficia empresas exportadoras.,1
4,2024-01-08,exemplo,Titulo 5,Setor bancário recua com aversão ao risco.,0


In [6]:
# 5. Pré-processamento de texto
import re, nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

stop_pt = set(stopwords.words("portuguese"))
PATTERN = r"[^A-Za-z\u00C0-\u00FF\s]"

def preprocess_text(t):
    t = re.sub(PATTERN, " ", str(t)).lower()
    toks = [w for w in t.split() if w and w not in stop_pt]
    return " ".join(toks)

# Usar a coluna 'texto'
df_news["clean_text"] = df_news["texto"].apply(preprocess_text)

print("✅ Pré-processamento concluído!")
display(df_news[["texto", "clean_text"]].head())



KeyboardInterrupt



In [ ]:
from pathlib import Path
print("Arquivos em PROC_PATH:")
for path in Path(PROC_PATH).iterdir():
    size_kb = path.stat().st_size / 1024
    print(f" - {path.name} ({size_kb:0.1f} KB)")